# Zajęcia 2: RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adrianstando/2026Z-PJATK-TEG/blob/main/02-rag/notebook.ipynb)

RAG (*Retrieval-Augmented Generation*): przed wywołaniem modelu wyszukuje się fragmenty dokumentów pasujące do pytania i dokleja je do promptu. Model odpowiada na podstawie tych fragmentów, a nie wyłącznie na podstawie wiedzy z treningu.

Zakres:
1. Korpus i chunking: rozmiar fragmentu, overlap.
2. Baza wektorowa Chroma i filtrowanie po metadanych.
3. Pełny pipeline RAG: pytanie → fragmenty → prompt → odpowiedź ze źródłami.
4. Wyszukiwanie hybrydowe: BM25 + embeddingi, łączenie rankingów (RRF).
5. Reranking cross-encoderem.
6. Ewaluacja: hit rate i MRR dla wyszukiwania, LLM jako sędzia dla odpowiedzi.

Komórki z tagiem `web:*` są pokazywane na prezentacji.

## 0. Instalacja i konfiguracja

In [1]:
# Colab / Deepnote / Codespaces: odkomentuj i uruchom raz
# %pip install -q openai python-dotenv numpy matplotlib chromadb rank-bm25 langchain-text-splitters sentence-transformers

Dostawcę wybiera się zmienną `LLM_PROVIDER` (plik `.env` albo w Colabie: ikona klucza → *Secrets*). Instrukcja konfiguracji dostępu: [00-dostep-do-llm](../00-dostep-do-llm/README.md).

| `LLM_PROVIDER` | Wymagania |
|---|---|
| `azure` | `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY`, deploymenty `gpt-4o-mini` i `text-embedding-3-small` |
| `openai` | `OPENAI_API_KEY` |
| `deepseek` | `DEEPSEEK_API_KEY`; tanie API, bez embeddingów |
| `ollama` | [Ollama](https://ollama.com) lokalnie albo w Colabie na darmowym GPU (komórka niżej), modele `gemma4:e4b`, `gemma3:1b`, `embeddinggemma` |
| `proxy` | uruchomione [proxy dla subskrypcji](../tools/teg-proxy/README.md) (Copilot, Claude, Codex); bez embeddingów |

Wszyscy dostawcy udostępniają API zgodne z OpenAI, więc dalszy kod jest wspólny. Gdy dostawca nie liczy embeddingów, służy do tego `EMBED_PROVIDER` (domyślnie `ollama`).

### Ollama w Colabie (darmowe GPU)

Colab daje za darmo kartę graficzną NVIDIA T4, na której działa Ollama. Najpierw *Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU*, potem komórka poniżej (odkomentowana). Na początek wystarczy mały `gemma3:1b`; większy `gemma4:e4b` zajmuje ok. 10 GB. Bez niego w *Secrets* ustawia się `OLLAMA_CHAT=gemma3:1b`.

In [2]:
# Colab z GPU: odkomentuj i uruchom raz na sesję
# !nvidia-smi                                   # czy GPU jest włączone
# !sudo apt-get -qq install -y zstd pciutils > /dev/null
# !curl -fsSL https://ollama.com/install.sh | sh
# import subprocess
# subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# !ollama pull gemma3:1b
# !ollama pull embeddinggemma
# !ollama pull gemma4:e4b                       # opcjonalnie, ~10 GB

In [3]:
import os

try:  # Colab: sekrety z panelu "Secrets"
    from google.colab import userdata
    for k in ["LLM_PROVIDER", "EMBED_PROVIDER", "OLLAMA_CHAT", "OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT",
              "AZURE_OPENAI_API_KEY", "DEEPSEEK_API_KEY", "ANTHROPIC_API_KEY"]:
        try:
            os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
except ImportError:  # lokalnie: plik .env
    from dotenv import load_dotenv
    load_dotenv(override=True)

from openai import OpenAI, AzureOpenAI

# chat: model rozumujący, classic: model bez rozumowania, embed: model embeddingów.
# W Azure to NAZWY DEPLOYMENTÓW utworzonych w AI Foundry.
MODELS = {
    "openai":   {"chat": "gpt-5-nano", "classic": "gpt-4.1-mini", "embed": "text-embedding-3-small"},
    "azure":    {"chat": "gpt-4o-mini", "classic": "gpt-4o-mini", "embed": "text-embedding-3-small"},
    "deepseek": {"chat": "deepseek-reasoner", "classic": "deepseek-chat", "embed": None},
    "ollama":   {"chat": os.getenv("OLLAMA_CHAT", "gemma4:e4b"), "classic": "gemma3:1b", "embed": "embeddinggemma"},
    "proxy":    {"chat": "copilot/gpt-5-mini", "classic": "copilot/gpt-5-mini", "embed": None},
}

def make_client(provider):
    if provider == "azure":
        return AzureOpenAI(api_version="2024-12-01-preview")  # AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY
    if provider == "deepseek":
        return OpenAI(base_url="https://api.deepseek.com", api_key=os.environ["DEEPSEEK_API_KEY"])
    if provider == "ollama":
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # klucz wymagany, ale ignorowany
    if provider == "proxy":
        return OpenAI(base_url="http://localhost:8787/v1", api_key="teg")
    return OpenAI()  # OPENAI_API_KEY

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")
EMBED_PROVIDER = os.getenv("EMBED_PROVIDER") or (PROVIDER if MODELS[PROVIDER]["embed"] else "ollama")

client = make_client(PROVIDER)
embed_client = client if EMBED_PROVIDER == PROVIDER else make_client(EMBED_PROVIDER)
CHAT, CLASSIC = MODELS[PROVIDER]["chat"], MODELS[PROVIDER]["classic"]
EMBED = MODELS[EMBED_PROVIDER]["embed"]
print(f"czat: {PROVIDER} ({CHAT}, klasyczny: {CLASSIC}) | embeddingi: {EMBED_PROVIDER} ({EMBED})")

czat: openai (gpt-5-nano, klasyczny: gpt-4.1-mini) | embeddingi: ollama (embeddinggemma)


Korpus to 15 krótkich dokumentów fikcyjnej firmy programistycznej (regulaminy HR, IT, dokumentacja produktu Lumen Sync) w folderze [`data/`](data/). Firma jest wymyślona, więc model nie zna odpowiedzi z treningu: bez wyszukiwania nie ma szans odpowiedzieć poprawnie. W Colabie komórka poniżej pobiera folder z repozytorium.

In [4]:
from pathlib import Path
import urllib.request, json as _json

DATA = Path("data")
if not DATA.exists():  # Colab: pobranie korpusu z GitHuba
    DATA.mkdir()
    api = "https://api.github.com/repos/adrianstando/2026Z-PJATK-TEG/contents/02-rag/data"
    for f in _json.load(urllib.request.urlopen(api)):
        urllib.request.urlretrieve(f["download_url"], DATA / f["name"])

## 1. Korpus

Każdy plik ma tytuł, linię metadanych (`dział`, `rok`) i treść. Metadane przydadzą się do filtrowania w sekcji 3.

In [5]:
def load_docs(folder):
    docs = []
    for path in sorted(Path(folder).glob("*.md")):
        title, meta, _, *body = path.read_text(encoding="utf-8").split("\n")
        meta = dict(kv.split(": ") for kv in meta.split(" | "))
        docs.append({"id": path.stem, "title": title.lstrip("# "), "dzial": meta["dział"],
                     "rok": int(meta["rok"]), "text": "\n".join(body).strip()})
    return docs

docs = load_docs(DATA)
print(f"dokumentów: {len(docs)}, znaków razem: {sum(len(d['text']) for d in docs)}")
for d in docs[:5]:
    print(f"  {d['id']:<24} {d['dzial']:<8} {d['rok']}  {len(d['text'])} znaków")

dokumentów: 15, znaków razem: 11158
  benefity                 HR       2025  731 znaków
  biuro                    Administracja 2025  730 znaków
  delegacje                Finanse  2026  784 znaków
  hasla-i-klucze           IT       2026  768 znaków
  lumen-sync-cennik        Produkt  2026  613 znaków


## 2. Chunking

Embedding całego dokumentu uśrednia jego treść: fragment o dietach w delegacji ginie w tekście o biletach i hotelach. Dlatego dokumenty dzieli się na chunki i liczy embedding dla każdego z nich.

- `chunk_size`: maksymalna długość fragmentu w znakach. Za małe chunki gubią kontekst („przysługuje połowa” – czego?), za duże mieszają tematy i zajmują miejsce w prompcie.
- `chunk_overlap`: ile znaków końca poprzedniego chunku powtarza się na początku następnego, żeby zdanie na granicy nie zostało rozcięte bez kontekstu.

`RecursiveCharacterTextSplitter` tnie najpierw po akapitach, potem po liniach, zdaniach i słowach, dopóki fragment nie zmieści się w limicie. Interaktywna wersja z suwakami jest na prezentacji.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc = next(d for d in docs if d["id"] == "delegacje")
for size, overlap in [(150, 0), (300, 50), (1000, 0)]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    parts = splitter.split_text(doc["text"])
    print(f"chunk_size={size:<5} overlap={overlap:<3} -> {len(parts)} chunków, długości {[len(p) for p in parts]}")
print("\nprzykład (150, 0), chunk 3:", repr(RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=0).split_text(doc["text"])[2]))

chunk_size=150   overlap=0   -> 7 chunków, długości [150, 52, 144, 53, 135, 145, 96]
chunk_size=300   overlap=50  -> 4 chunków, długości [203, 198, 135, 242]
chunk_size=1000  overlap=0   -> 1 chunków, długości [784]

przykład (150, 0), chunk 3: 'Dieta za dobę podróży krajowej wynosi 45 zł. Jeśli podróż trwa od 8 do 12 godzin, przysługuje połowa diety. Dieta jest pomniejszana o zapewnione'


Fragment bez kontekstu („przysługuje połowa diety”) nie mówi, o jaką dietę chodzi. Prosty sposób na poprawę: dokleić do każdego chunku tytuł dokumentu. Metadane (`doc`, `dzial`, `rok`) zapisuje się przy chunku, żeby później podać źródło odpowiedzi i filtrować.

In [7]:
def make_chunks(docs, size=300, overlap=50):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks = []
    for d in docs:
        for i, part in enumerate(splitter.split_text(d["text"])):
            chunks.append({"id": f"{d['id']}#{i}", "text": f"{d['title']}\n{part}",
                           "meta": {"doc": d["id"], "dzial": d["dzial"], "rok": d["rok"]}})
    return chunks

chunks = make_chunks(docs)
print(len(chunks), "chunków")
print(chunks[12]["id"], chunks[12]["meta"])
print(chunks[12]["text"])

60 chunków
hasla-i-klucze#0 {'doc': 'hasla-i-klucze', 'dzial': 'IT', 'rok': 2026}
Hasła, klucze sprzętowe i uwierzytelnianie
Hasło do konta firmowego musi mieć co najmniej 14 znaków. Hasła nie wymienia się okresowo; zmiana jest wymagana tylko po podejrzeniu wycieku. Do przechowywania haseł służy firmowy menedżer haseł.


## 3. Baza wektorowa: Chroma

[Chroma](https://docs.trychroma.com) przechowuje teksty, embeddingi i metadane, a przy zapytaniu zwraca najbliższe wektory. Embeddingi liczy się tą samą funkcją `embed()` co na zajęciach 1, więc działa każdy dostawca. `chromadb.Client()` trzyma dane w pamięci; `PersistentClient(path=...)` zapisuje je na dysk.

`hnsw:space = cosine`: Chroma zwraca odległość cosinusową (`1 − cos`), więc mniej znaczy bliżej.

In [8]:
import numpy as np
import chromadb

_cache = {}
def embed(texts):
    """Embeddingi dla listy tekstów (z cache)."""
    todo = [t for t in texts if t not in _cache]
    if todo:
        for t, d in zip(todo, embed_client.embeddings.create(model=EMBED, input=todo).data):
            _cache[t] = d.embedding
    return [_cache[t] for t in texts]

def build_collection(chunks, name="firma"):
    db = chromadb.Client()
    if name in [c.name for c in db.list_collections()]:
        db.delete_collection(name)
    col = db.create_collection(name, metadata={"hnsw:space": "cosine"})
    col.add(ids=[c["id"] for c in chunks], documents=[c["text"] for c in chunks],
            metadatas=[c["meta"] for c in chunks], embeddings=embed([c["text"] for c in chunks]))
    return col

collection = build_collection(chunks)
print("w kolekcji:", collection.count())

w kolekcji: 60


In [9]:
def vector_search(query, k=3, where=None):
    res = collection.query(query_embeddings=embed([query]), n_results=k, where=where)
    return list(zip(res["ids"][0], res["distances"][0], res["metadatas"][0]))

for cid, dist, meta in vector_search("Ile dni w tygodniu można pracować zdalnie?"):
    print(f"{dist:.3f}  {cid:<22} rok={meta['rok']}")

0.195  praca-zdalna-2024#1    rok=2024
0.253  praca-zdalna-2026#1    rok=2026
0.323  praca-zdalna-2026#3    rok=2026


Najbliższy fragment pochodzi z **nieaktualnej** polityki z 2024 roku (dwa dni zamiast trzech). Embedding nie wie, która wersja obowiązuje: oba teksty mówią o tym samym. Tu pomagają metadane.

### Filtrowanie po metadanych

Parametr `where` zawęża wyszukiwanie do chunków spełniających warunek, przed wyborem `k` najlepszych (nie trzeba pobierać `3·k` wyników i filtrować w Pythonie). Operatory: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, łączone przez `$and` / `$or`.

In [10]:
q = "Ile dni w tygodniu można pracować zdalnie?"
for where in [{"rok": {"$gte": 2026}}, {"$and": [{"dzial": "HR"}, {"rok": 2024}]}]:
    print("where =", where)
    for cid, dist, meta in vector_search(q, where=where):
        print(f"   {dist:.3f}  {cid:<22} {meta['dzial']} {meta['rok']}")

where = {'rok': {'$gte': 2026}}
   0.253  praca-zdalna-2026#1    HR 2026
   0.323  praca-zdalna-2026#3    HR 2026
   0.383  praca-zdalna-2026#4    HR 2026
where = {'$and': [{'dzial': 'HR'}, {'rok': 2024}]}
   0.195  praca-zdalna-2024#1    HR 2024
   0.340  praca-zdalna-2024#2    HR 2024
   0.458  praca-zdalna-2024#0    HR 2024


W praktyce filtr wynika z kontekstu użytkownika (dział, uprawnienia, język) albo z oznaczenia dokumentów jako archiwalnych. Można też poprosić LLM o wyciągnięcie filtra z pytania („według zasad z 2024 roku” → `{"rok": 2024}`); to wariant agentowy z zajęć 3.

**Ćwiczenie.** Wyszukaj „Co oznacza kod błędu?” bez filtra i z filtrem `{"dzial": "Produkt"}`. Ile wyników z działu IT znika?

## 4. Pipeline RAG

Cztery kroki: pytanie → embedding → `k` najbliższych chunków → prompt z kontekstem → odpowiedź. Prompt wymaga odpowiedzi wyłącznie na podstawie kontekstu i podania źródeł, a przy braku informacji: „nie wiem”.

In [11]:
SYSTEM = """Odpowiadasz na pytania pracowników na podstawie fragmentów dokumentów firmy.
Korzystasz wyłącznie z podanego kontekstu. Jeśli kontekst nie zawiera odpowiedzi, piszesz: "Nie wiem".
Odpowiedź: 1-2 zdania po polsku, na końcu identyfikatory źródeł w nawiasach kwadratowych."""

def rag(question, k=3, where={"rok": {"$gte": 2025}}):
    hits = vector_search(question, k=k, where=where)
    texts = collection.get(ids=[cid for cid, _, _ in hits])
    context = "\n\n".join(f"[{i}] {t}" for i, t in zip(texts["ids"], texts["documents"]))
    msg = client.chat.completions.create(model=CHAT, messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Kontekst:\n{context}\n\nPytanie: {question}"},
    ]).choices[0].message.content
    return msg, [cid for cid, _, _ in hits]

for q in ["Ile dni w tygodniu można pracować zdalnie?", "Czy firma dopłaca do rachunków za prąd przy pracy z domu?",
          "Ile wynosi pensja prezesa?"]:
    answer, sources = rag(q)
    print(f"Q: {q}\nA: {answer}\n   kontekst: {sources}\n")

Q: Ile dni w tygodniu można pracować zdalnie?
A: Można pracować zdalnie do trzech dni w tygodniu [praca-zdalna-2026#1].
   kontekst: ['praca-zdalna-2026#1', 'praca-zdalna-2026#3', 'praca-zdalna-2026#4']



Q: Czy firma dopłaca do rachunków za prąd przy pracy z domu?
A: Tak. Firma wypłaca ryczałt 250 zł brutto na pokrycie kosztów internetu i energii za każdy miesiąc, w którym pracownik pracował zdalnie co najmniej 8 dni, i jest on wypłacany razem z wynagrodzeniem. [praca-zdalna-2026#2]
   kontekst: ['praca-zdalna-2026#2', 'sprzet#1', 'praca-zdalna-2026#4']



Q: Ile wynosi pensja prezesa?
A: Nie wiem [delegacje#1][delegacje#2][praca-zdalna-2026#2]
   kontekst: ['delegacje#2', 'praca-zdalna-2026#2', 'delegacje#1']



Trzecie pytanie sprawdza, czy model nie zmyśla, gdy kontekst nie zawiera odpowiedzi. Dla porównania: to samo pytanie o pracę zdalną bez kontekstu.

In [12]:
q = "Ile dni w tygodniu można pracować zdalnie w firmie produkującej Lumen Sync?"
print(client.chat.completions.create(model=CHAT, messages=[{"role": "user", "content": q + " Odpowiedz w 1-2 zdaniach."}]).choices[0].message.content)

Nie mam dostępu do wewnętrznych zasad tej firmy produkującej Lumen Sync. Zwykle firmy stosują model hybrydowy — najczęściej 2–3 dni zdalnie w tygodniu, ale dokładna liczba zależy od stanowiska i projektu.


**Ćwiczenie.** Zmień `k` na 1 i 8. Przy jakim pytaniu `k=1` nie wystarcza? Co się dzieje z długością promptu przy `k=8`?

## 5. Wyszukiwanie hybrydowe: BM25

Embeddingi dobrze łapią parafrazy („stary komputer” → „wymiana laptopa”), gorzej rzadkie identyfikatory: kody błędów, numery, nazwy własne. BM25 to klasyczny ranking słów kluczowych: punktuje dokumenty zawierające słowa z zapytania, tym wyżej, im rzadsze jest słowo w korpusie (IDF), z karą za długość dokumentu.

Polski ma bogatą fleksję („urlop”, „urlopu”, „urlopów”), więc słowa przycina się do 6 znaków (prymitywny stemming); kody z cyframi zostają w całości. Lepsze wyniki daje lematyzator (np. spaCy `pl_core_news_sm`).

In [13]:
import re
from rank_bm25 import BM25Okapi

STOP = set("a i w z na do o od po się że to jest są nie co jak ile czy dla przez lub albo oraz też mogę".split())

def tokenize(text):
    words = re.findall(r"\w+(?:-\d+)?", text.lower())
    return [w if any(ch.isdigit() for ch in w) else w[:6] for w in words if w not in STOP]

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])
CHUNK_IDS = [c["id"] for c in chunks]

def bm25_search(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    best = np.argsort(-scores)[:k]
    return [(CHUNK_IDS[i], float(scores[i])) for i in best]

print(tokenize("Zgubiłem klucz YubiKey, błąd VPN-719"))
for q in ["Co oznacza błąd LS-409?", "Ile pieniędzy mogę wydać na kursy i konferencje?"]:
    print(f"\nQ: {q}")
    print("  wektory:", [cid for cid, _, _ in vector_search(q, where=None)])
    print("  BM25:   ", [cid for cid, _ in bm25_search(q)])

['zgubił', 'klucz', 'yubike', 'błąd', 'vpn-719']

Q: Co oznacza błąd LS-409?


  wektory: ['lumen-sync-kody-bledow#1', 'lumen-sync-kody-bledow#0', 'lumen-sync-kody-bledow#2']
  BM25:    ['lumen-sync-limity-api#2', 'vpn#2', 'lumen-sync-kody-bledow#1']

Q: Ile pieniędzy mogę wydać na kursy i konferencje?


  wektory: ['delegacje#2', 'benefity#2', 'lumen-sync-limity-api#1']
  BM25:    ['benefity#2', 'biuro#1', 'benefity#1']


### Łączenie rankingów: Reciprocal Rank Fusion

Wyniki BM25 (0–20) i odległości cosinusowe (0–2) mają różne skale, więc nie sumuje się ich wprost. RRF patrzy tylko na pozycje: chunk dostaje `1 / (60 + pozycja)` z każdej listy, a punkty się sumuje. Stała 60 pochodzi z oryginalnej pracy (Cormack i in., 2009) i spłaszcza różnice między czołowymi pozycjami. Chunk wysoko na obu listach wygrywa z chunkiem, który jest pierwszy tylko na jednej.

In [14]:
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for pos, cid in enumerate(ranking, start=1):
            scores[cid] = scores.get(cid, 0) + 1 / (k + pos)
    return sorted(scores, key=scores.get, reverse=True)

def hybrid_search(query, k=3, depth=20):
    vec = [cid for cid, _, _ in vector_search(query, k=depth, where=None)]
    kw = [cid for cid, _ in bm25_search(query, k=depth)]
    return rrf([vec, kw])[:k]

q = "Ile pieniędzy mogę wydać na kursy i konferencje?"
print("wektory: ", [cid for cid, _, _ in vector_search(q, where=None)])
print("BM25:    ", [cid for cid, _ in bm25_search(q)])
print("RRF:     ", hybrid_search(q))

wektory:  ['delegacje#2', 'benefity#2', 'lumen-sync-limity-api#1']
BM25:     ['benefity#2', 'biuro#1', 'benefity#1']
RRF:      ['benefity#2', 'delegacje#2', 'biuro#1']


## 6. Reranking

Model embeddingów koduje pytanie i chunk osobno (bi-encoder), więc wektory chunków można policzyć raz, z góry. Cross-encoder czyta pytanie i chunk razem i zwraca jedną liczbę: trafność. Jest dokładniejszy, ale wolny (jedno wywołanie modelu na parę), dlatego ocenia tylko kilkanaście kandydatów z pierwszego etapu.

Model: [`sdadas/polish-reranker-base-ranknet`](https://huggingface.co/sdadas/polish-reranker-base-ranknet), trenowany na polskich danych, ok. 500 MB. Pierwsze uruchomienie pobiera model (w Colabie zwykle kilkanaście sekund, przy wolnym łączu do 2 minut); później jest w cache. Działa na CPU.

In [15]:
import os, time, warnings
os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS="1", HF_HUB_VERBOSITY="error")  # mniej komunikatów przy pobieraniu
warnings.filterwarnings("ignore")
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
from sentence_transformers import CrossEncoder

t0 = time.time()
reranker = CrossEncoder("sdadas/polish-reranker-base-ranknet")
print(f"model wczytany w {time.time() - t0:.0f} s")

TEXT = {c["id"]: c["text"] for c in chunks}

def rerank(query, candidates, k=3):
    scores = reranker.predict([(query, TEXT[cid]) for cid in candidates])
    order = np.argsort(-scores)[:k]
    return [(candidates[i], float(scores[i])) for i in order]

model wczytany w 3 s


In [16]:
q = "Mój komputer jest już stary, kiedy dostanę nowy?"
candidates = hybrid_search(q, k=10)
print("hybryda, top 5:", candidates[:5])
t0 = time.time()
for cid, s in rerank(q, candidates, k=5):
    print(f"  {s:6.3f}  {cid:<22} (było miejsce {candidates.index(cid) + 1})")
print(f"reranking 10 kandydatów: {time.time() - t0:.2f} s")

hybryda, top 5: ['hasla-i-klucze#2', 'sprzet#0', 'praca-zdalna-2026#1', 'sprzet#1', 'benefity#0']


   0.757  sprzet#0               (było miejsce 2)
   0.578  sprzet#1               (było miejsce 4)
   0.452  sprzet#3               (było miejsce 10)
   0.322  onboarding#0           (było miejsce 8)
   0.315  onboarding#2           (było miejsce 6)
reranking 10 kandydatów: 2.70 s


## 7. Ewaluacja wyszukiwania: hit rate i MRR

Bez zestawu testowego nie da się stwierdzić, czy zmiana (rozmiar chunku, BM25, reranking) poprawia system. Minimum: kilkanaście pytań z oznaczonym dokumentem, w którym jest odpowiedź.

- **hit rate@k**: odsetek pytań, dla których właściwy dokument jest w pierwszych `k` wynikach,
- **MRR** (*mean reciprocal rank*): średnia z `1 / pozycja` właściwego dokumentu; 1.0 = zawsze na pierwszym miejscu, 0.5 = średnio na drugim.

Pytania celowo zawierają parafrazy, kody błędów i jedną pułapkę z dwiema wersjami polityki.

In [17]:
EVAL = [
    ("Ile dni w tygodniu można pracować zdalnie?", "praca-zdalna-2026", "Do trzech dni w tygodniu."),
    ("Co oznacza błąd LS-409?", "lumen-sync-kody-bledow", "Konflikt wersji pliku zmienionego na innym urządzeniu."),
    ("Jak naprawić błąd VPN-719?", "vpn", "Odnowić certyfikat w portalu samoobsługowym i ponownie zaimportować konfigurację."),
    ("Ile wynosi dieta za dobę w podróży służbowej po Polsce?", "delegacje", "45 zł."),
    ("Z jakim wyprzedzeniem trzeba złożyć wniosek o dwutygodniowy urlop?", "urlopy", "Co najmniej 14 dni wcześniej."),
    ("Ile kosztuje plan Zespół?", "lumen-sync-cennik", "39 zł netto za użytkownika miesięcznie przy płatności rocznej."),
    ("Ile zapytań na minutę obsługuje API w najtańszym planie?", "lumen-sync-limity-api", "60 zapytań na minutę (plan Start)."),
    ("Jak długo trzymamy logi z adresami e-mail?", "ochrona-danych", "30 dni."),
    ("W jakim czasie dyżurny reaguje na krytyczną awarię?", "zglaszanie-awarii", "W ciągu 15 minut, całą dobę."),
    ("Zgubiłem YubiKey, co teraz?", "hasla-i-klucze", "Zgłosić na helpdesk jako P2; IT zablokuje klucz i wyda zapasowy w ciągu dnia roboczego."),
    ("Mój komputer jest już stary, kiedy dostanę nowy?", "sprzet", "Laptop wymienia się co trzy lata."),
    ("Jak zarezerwować salę na spotkanie?", "biuro", "Dodać salę jako uczestnika spotkania w kalendarzu."),
    ("Ile pieniędzy mogę wydać na kursy i konferencje?", "benefity", "3000 zł rocznie."),
    ("Kto pomaga nowej osobie w pierwszych miesiącach?", "onboarding", "Opiekun wdrożenia (buddy), przez trzy miesiące."),
    ("Czy dostanę dopłatę do rachunków za prąd, gdy pracuję z domu?", "praca-zdalna-2026", "Tak, 250 zł brutto miesięcznie przy co najmniej 8 dniach pracy zdalnej."),
    ("Co zrobić, gdy wysłałem dane klienta do złej osoby?", "ochrona-danych", "Zgłosić inspektorowi ochrony danych najpóźniej w ciągu 24 godzin."),
]
print(len(EVAL), "pytań")

16 pytań


In [18]:
def doc_rank(ranked_chunk_ids, gold_doc):
    """Pozycja pierwszego chunku z właściwego dokumentu (None, jeśli go nie ma)."""
    for pos, cid in enumerate(ranked_chunk_ids, start=1):
        if cid.split("#")[0] == gold_doc:
            return pos
    return None

def evaluate(search_fn, k=3):
    ranks = [doc_rank(search_fn(q), gold) for q, gold, _ in EVAL]
    hit = np.mean([r is not None and r <= k for r in ranks])
    mrr = np.mean([1 / r if r else 0 for r in ranks])
    return hit, mrr

METHODS = {
    "wektory":           lambda q: [cid for cid, _, _ in vector_search(q, k=10, where=None)],
    "BM25":              lambda q: [cid for cid, _ in bm25_search(q, k=10)],
    "hybryda (RRF)":     lambda q: hybrid_search(q, k=10),
    "hybryda + rerank":  lambda q: [cid for cid, _ in rerank(q, hybrid_search(q, k=10), k=10)],
    "wektory + filtr":   lambda q: [cid for cid, _, _ in vector_search(q, k=10, where={"rok": {"$gte": 2025}})],
}
print(f"{'metoda':<20}{'hit@1':>7}{'hit@3':>7}{'MRR':>7}")
for name, fn in METHODS.items():
    h1, mrr = evaluate(fn, k=1)
    h3, _ = evaluate(fn, k=3)
    print(f"{name:<20}{h1:>7.2f}{h3:>7.2f}{mrr:>7.3f}")

metoda                hit@1  hit@3    MRR


wektory                0.88   1.00  0.938
BM25                   0.69   0.94  0.792
hybryda (RRF)          0.81   1.00  0.906


hybryda + rerank       1.00   1.00  1.000
wektory + filtr        0.94   1.00  0.969


Szesnaście pytań to mało: jedno pytanie zmienia hit@1 o 0.06. Wnioski z takiej tabeli są wskazówką, a nie dowodem. Na małym, uporządkowanym korpusie dobre embeddingi same radzą sobie dobrze; BM25 i reranking zyskują na znaczeniu przy tysiącach dokumentów, kodach i nazwach własnych. Metryka sprawdza tylko dokument, a nie to, czy chunk zawiera odpowiedź.

### Wpływ rozmiaru chunku

In [19]:
print(f"{'chunk_size':>10}{'overlap':>9}{'chunków':>9}{'hit@1':>7}{'MRR':>7}")
for size, overlap in [(150, 0), (300, 50), (600, 100), (1200, 0)]:
    chunks_v = make_chunks(docs, size, overlap)
    collection = build_collection(chunks_v)
    h1, mrr = evaluate(METHODS["wektory"], k=1)
    print(f"{size:>10}{overlap:>9}{len(chunks_v):>9}{h1:>7.2f}{mrr:>7.3f}")
collection = build_collection(chunks)  # powrót do ustawień 300/50

chunk_size  overlap  chunków  hit@1    MRR


       150        0      109   0.75  0.875


       300       50       60   0.88  0.938


       600      100       33   0.88  0.938


      1200        0       15   0.88  0.938


Przy `1200` każdy dokument to jeden chunk: wyszukiwanie dokumentu jest łatwe, ale do promptu trafia cały dokument zamiast potrzebnego fragmentu (więcej tokenów, więcej szumu dla modelu).

## 8. Ewaluacja odpowiedzi: LLM jako sędzia

Hit rate ocenia wyszukiwanie, nie odpowiedź. Odpowiedź ocenia się drugim wywołaniem modelu z instrukcją oceny w dwóch wymiarach:
- **wierność** (*faithfulness*): czy każde twierdzenie odpowiedzi wynika z kontekstu (brak halucynacji),
- **poprawność**: czy odpowiedź zgadza się z odpowiedzią wzorcową.

Tak działają biblioteki [RAGAS](https://docs.ragas.io) i [DeepEval](https://deepeval.com) (gotowe metryki *faithfulness*, *answer relevancy*, *context precision/recall*); tu w kilkunastu liniach, żeby było widać, co jest w środku. Sędzia też się myli, więc sprawdza się go na przypadkach kontrolnych o znanej ocenie, a część ocen ręcznie.

In [20]:
JUDGE = """Oceniasz odpowiedź systemu RAG. Zwróć wyłącznie JSON:
{"faithful": 0 albo 1, "correct": 0 albo 1, "reason": "jedno zdanie"}
faithful = 1, jeśli każde twierdzenie odpowiedzi wynika z kontekstu.
correct = 1, jeśli odpowiedź zawiera informację z odpowiedzi wzorcowej (inne sformułowanie jest w porządku)."""

def rag_with_context(question, k=3):
    answer, ids = rag(question, k=k)
    return answer, "\n\n".join(collection.get(ids=ids)["documents"])

def judge(question, answer, context, reference):
    out = client.chat.completions.create(model=CHAT, response_format={"type": "json_object"}, messages=[
        {"role": "system", "content": JUDGE},
        {"role": "user", "content": f"Pytanie: {question}\nKontekst:\n{context}\n\nOdpowiedź wzorcowa: {reference}\nOdpowiedź systemu: {answer}"},
    ])
    return json.loads(out.choices[0].message.content)

In [21]:
import json

results = []
for q, gold, reference in EVAL[:8]:
    answer, context = rag_with_context(q)
    verdict = judge(q, answer, context, reference)
    results.append(verdict)
    print(f"F={verdict['faithful']} C={verdict['correct']}  {q[:48]:<48} | {verdict['reason'][:70]}")
print(f"\nwierność: {np.mean([r['faithful'] for r in results]):.2f}   poprawność: {np.mean([r['correct'] for r in results]):.2f}")

F=1 C=1  Ile dni w tygodniu można pracować zdalnie?       | Z kontekstu wynika, że praca zdalna do trzech dni w tygodniu.


F=1 C=1  Co oznacza błąd LS-409?                          | Odpowiedź systemu trafnie wskazuje, że LS-409 to konflikt wersji pliku


F=1 C=1  Jak naprawić błąd VPN-719?                       | Odpowiedź systemu powiela kluczowy krok wzorcowej (odnowić certyfikat 


F=1 C=1  Ile wynosi dieta za dobę w podróży służbowej po  | jedno zdanie


F=1 C=1  Z jakim wyprzedzeniem trzeba złożyć wniosek o dw | Odpowiedź systemu zgadza się z kontekstem, mówiąc o co najmniej 14 dni


F=1 C=1  Ile kosztuje plan Zespół?                        | Cena 39 zł netto za użytkownika miesięcznie przy płatności rocznej i i


F=1 C=1  Ile zapytań na minutę obsługuje API w najtańszym | Tak; odpowiedź systemu wiernie odzwierciedla kontekst, że najtańszy pl


F=1 C=1  Jak długo trzymamy logi z adresami e-mail?       | Odpowiedź systemu bezpośrednio odzwierciedla kontekst, że logi z e-mai

wierność: 1.00   poprawność: 1.00


Same oceny 1/1 nie mówią, czy sędzia w ogóle umie odrzucić złą odpowiedź. Dlatego dodaje się przypadki kontrolne o znanej ocenie:
1. halucynacja: do poprawnej odpowiedzi dopisany fakt, którego nie ma w kontekście (oczekiwane `faithful=0`),
2. zły kontekst: wyszukiwanie zwróciło nieaktualną politykę z 2024 roku, a odpowiedź wiernie ją powtarza (oczekiwane `faithful=1`, `correct=0`).

In [22]:
def doc_context(doc_id):
    return "\n\n".join(collection.get(where={"doc": doc_id})["documents"])

CONTROLS = [
    ("halucynacja", "Ile wynosi dieta za dobę w podróży służbowej po Polsce?", doc_context("delegacje"),
     "Dieta wynosi 45 zł za dobę, a w weekendy i święta 90 zł.", "45 zł.", (0, 1)),
    ("zły kontekst", "Ile dni w tygodniu można pracować zdalnie?", doc_context("praca-zdalna-2024"),
     "Można pracować zdalnie najwyżej dwa dni w tygodniu.", "Do trzech dni w tygodniu.", (1, 0)),
]
for name, q, context, answer, reference, (exp_f, exp_c) in CONTROLS:
    v = judge(q, answer, context, reference)
    ok = "OK" if (v["faithful"], v["correct"]) == (exp_f, exp_c) else "BŁĄD SĘDZIEGO"
    print(f"{name:<13} oczekiwane F={exp_f} C={exp_c} | sędzia F={v['faithful']} C={v['correct']}  {ok}")
    print(f"{'':<13} {v['reason'][:110]}")

halucynacja   oczekiwane F=0 C=1 | sędzia F=0 C=1  OK
              Odpowiedź prawidłowo podaje 45 zł jako dietę za dobę w podróży krajowej, ale dodatkowe stwierdzenie o 90 zł na


zły kontekst  oczekiwane F=1 C=0 | sędzia F=1 C=0  OK
              Odpowiedź systemu zgadza się z kontekstem (dwa dni), ale nie odpowiada wzorcowej odpowiedzi.


**Ćwiczenie.** Podmień w `rag()` wyszukiwanie wektorowe na `hybrid_search` + `rerank` i uruchom sędziego jeszcze raz. Czy wynik się zmienił? Na których pytaniach?

## Zadanie

Zadanie po zajęciach: [zadanie.ipynb](zadanie.ipynb) ([Colab](https://colab.research.google.com/github/adrianstando/2026Z-PJATK-TEG/blob/main/02-rag/zadanie.ipynb)).